In [1]:
from math import pi
import numpy as np
import scipy as sp
from qiskit.quantum_info import Pauli
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.circuit import Parameter
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import SciPyOptimizer
from qiskit.primitives import Sampler # new
from qiskit.quantum_info import SparsePauliOp,PauliList
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.primitives import Estimator # new
import time
import pandas as pd
from qiskit.providers.basic_provider import BasicProvider #new
from multiprocessing import Pool
import multiprocessing as mp
from qiskit_algorithms.optimizers import SciPyOptimizer
import scipy as sp
import os
#from sko.SA import SA

In [3]:
def create_ham_str(nqubits):

    # Create a list of terms for the hamiltonian (open boundary conditions)
    # Inputs: nqubits (int), number of qubits
    # Outputs: ham (list), hamiltonian 

    ham = []

    for i in range(nqubits-1):

        term = ''

        for j in range(nqubits-i-2):

            term += 'I'

        for j in range(nqubits-i-2,nqubits-i):

            term += 'Y'  # Choose a Pauli matrix, i.e., X, Y or Z

        for j in range(nqubits-i,nqubits):

            term += 'I'

        ham.append(term)

    return ham



def evaluate_expectation(parameters_values):
    # Function to evaluate the expectation value of the Hamiltonian for a given set of parameters
    # Inputs: parameter_values (ndarray), parameter values
    # Outputs: result (float), energy value

    value_dict = dict(zip(ansatz.parameters, parameters_values))
    pars =  list(value_dict.values())
    expectation_value = estimator.run(ansatz,qubit_op,pars).result().values
    return np.real(expectation_value)

In [6]:
import numpy as np
import math
import random
import os

class WhaleOptimizationAlgorithm:
    def __init__(self, hunting_party=5, spiral_param=1, min_values=[-500]*10, max_values=[500]*10, 
                 tol=1e-1, target_function=None, dimension=10, qubits=1):
        self.hunting_party = hunting_party
        self.spiral_param = spiral_param
        self.min_values = min_values
        self.max_values = max_values
        self.tol = tol
        self.target_function = target_function
        self.dimension = dimension
        self.qubits = qubits
        self.func_eval_count = 0

    def evaluate(self, params):
        self.func_eval_count += 1
        return self.target_function(params)

    def initial_position(self):
        position = np.zeros((self.hunting_party, len(self.min_values) + 1))
        for i in range(self.hunting_party):
            for j in range(len(self.min_values)):
                position[i, j] = random.uniform(self.min_values[j], self.max_values[j])
            position[i, -1] = self.evaluate(position[i, 0:position.shape[1]-1])
        return position

    def leader_position(self):
        leader = np.zeros((1, self.dimension + 1))
        for j in range(self.dimension):
            leader[0, j] = 0.0
        leader[0, -1] = self.evaluate(leader[0, 0:leader.shape[1]-1])
        return leader

    def update_leader(self, position, leader):
        for i in range(position.shape[0]):
            if leader[0, -1] > position[i, -1]:
                for j in range(position.shape[1]):
                    leader[0, j] = position[i, j]
        return leader

    def update_position(self, position, leader, a_linear_component=2, b_linear_component=1):
        for i in range(position.shape[0]):
            r1_leader = int.from_bytes(os.urandom(8), byteorder="big") / ((1 << 64) - 1)
            r2_leader = int.from_bytes(os.urandom(8), byteorder="big") / ((1 << 64) - 1)
            a_leader = 2 * a_linear_component * r1_leader - a_linear_component
            c_leader = 2 * r2_leader
            p = int.from_bytes(os.urandom(8), byteorder="big") / ((1 << 64) - 1)
            for j in range(len(self.min_values)):
                if p < 0.5:
                    if abs(a_leader) >= 1:
                        rand = int.from_bytes(os.urandom(8), byteorder="big") / ((1 << 64) - 1)
                        rand_leader_index = math.floor(position.shape[0] * rand)
                        x_rand = position[rand_leader_index, :]
                        distance_x_rand = abs(c_leader * x_rand[j] - position[i, j])
                        position[i, j] = np.clip(x_rand[j] - a_leader * distance_x_rand, self.min_values[j], self.max_values[j])
                    elif abs(a_leader) < 1:
                        distance_leader = abs(c_leader * leader[0, j] - position[i, j])
                        position[i, j] = np.clip(leader[0, j] - a_leader * distance_leader, self.min_values[j], self.max_values[j])
                elif p >= 0.5:
                    distance_leader = abs(leader[0, j] - position[i, j])
                    rand = int.from_bytes(os.urandom(8), byteorder="big") / ((1 << 64) - 1)
                    m_param = (b_linear_component - 1) * rand + 1
                    position[i, j] = np.clip(distance_leader * math.exp(self.spiral_param * m_param) * math.cos(m_param * 2 * math.pi) + leader[0, j], self.min_values[j], self.max_values[j])
            position[i, -1] = self.evaluate(position[i, 0:position.shape[1] - 1])
        return position

    def run(self):
        position = self.initial_position()
        leader = self.leader_position()
        iteration = 0
        while np.abs(leader[0, -1] + self.qubits - 1) >= self.tol:
            if iteration % 100 == 1:
                print(f"Iteration: {iteration}, Best Fitness: {leader[0, -1]}, Evaluations: {self.func_eval_count}")
            a_linear_component = 2 - iteration * (2 / 50)
            b_linear_component = -1 + iteration * (-1 / 50)
            leader = self.update_leader(position, leader)
            position = self.update_position(position, leader, a_linear_component=a_linear_component, b_linear_component=b_linear_component)
            iteration += 1
        #print("Optimal Leader Position:", leader)
        print(f"Iteration: {iteration}, Best Fitness: {leader[0, -1]}, Evaluations: {self.func_eval_count}")
        return leader, self.func_eval_count


In [ ]:
min_qubits = 3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    woa = WhaleOptimizationAlgorithm(hunting_party=10, spiral_param=1, min_values=[-50]*dimension, max_values=[50]*dimension, 
                                     tol=1e-1, target_function=evaluate_expectation, dimension=dimension, qubits=qubits)
    best_leader, eval_count = woa.run()
    #print('Best leader:', best_leader)
    print('Function evaluations:', eval_count)

ansatz_num_parameters= 12
Iteration: 1, Best Fitness: -0.65625, Evaluations: 21


C:\Users\petre\AppData\Local\Temp\ipykernel_24080\3640854372.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  position[i, -1] = self.evaluate(position[i, 0:position.shape[1]-1])
C:\Users\petre\AppData\Local\Temp\ipykernel_24080\3640854372.py:35: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  leader[0, -1] = self.evaluate(leader[0, 0:leader.shape[1]-1])
C:\Users\petre\AppData\Local\Temp\ipykernel_24080\3640854372.py:68: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

Iteration: 101, Best Fitness: -1.875, Evaluations: 1021
Optimal Leader Position: [[  0.12436497  -0.15954442  -2.75758599   6.95176142   1.92958975
   -0.79975161   1.1344343    1.34145437  -2.27041291 -11.61931152
   -4.39229537   0.82802172  -1.90625   ]]
Iteration: 102, Best Fitness: -1.90625, Evaluations: 1031
Best leader: [[  0.12436497  -0.15954442  -2.75758599   6.95176142   1.92958975
   -0.79975161   1.1344343    1.34145437  -2.27041291 -11.61931152
   -4.39229537   0.82802172  -1.90625   ]]
Function evaluations: 1031
ansatz_num_parameters= 16
Iteration: 1, Best Fitness: -1.0, Evaluations: 21
Iteration: 101, Best Fitness: -2.8125, Evaluations: 1021
Iteration: 201, Best Fitness: -2.8125, Evaluations: 2021
Iteration: 301, Best Fitness: -2.8125, Evaluations: 3021
Optimal Leader Position: [[-29.32395229  -6.35301165  -0.60875444  -6.46225176   3.76715618
   -1.04866933   2.48923358 -32.45053808   1.47225786  -7.13160937
    5.14073468  -1.85558359  -7.7683853   -0.8689642    4.808

KeyboardInterrupt: 

In [11]:
import numpy as np
import math
import random
import os

class WhaleOptimizationAlgorithm:
    def __init__(self, hunting_party=5, spiral_param=1, min_values=[-500]*10, max_values=[500]*10, 
                 tol=1e-1, target_function=None, dimension=10, qubits=1):
        self.hunting_party = hunting_party
        self.spiral_param = spiral_param
        self.min_values = min_values
        self.max_values = max_values
        self.tol = tol
        self.target_function = target_function
        self.dimension = dimension
        self.qubits = qubits
        self.func_eval_count = 0

    def evaluate(self, params):
        self.func_eval_count += 1
        return self.target_function(params)

    def initial_position(self):
        position = np.zeros((self.hunting_party, len(self.min_values) + 1))
        for i in range(self.hunting_party):
            for j in range(len(self.min_values)):
                position[i, j] = random.uniform(self.min_values[j], self.max_values[j])
            position[i, -1] = self.evaluate(position[i, 0:position.shape[1]-1])
        return position

    def leader_position(self):
        leader = np.zeros((1, self.dimension + 1))
        for j in range(self.dimension):
            leader[0, j] = 0.0
        leader[0, -1] = self.evaluate(leader[0, 0:leader.shape[1]-1])
        return leader

    def update_leader(self, position, leader):
        for i in range(position.shape[0]):
            if leader[0, -1] > position[i, -1]:
                for j in range(position.shape[1]):
                    leader[0, j] = position[i, j]
        return leader

    def update_position(self, position, leader, a_linear_component=2, b_linear_component=1):
        for i in range(position.shape[0]):
            r1_leader = int.from_bytes(os.urandom(8), byteorder="big") / ((1 << 64) - 1)
            r2_leader = int.from_bytes(os.urandom(8), byteorder="big") / ((1 << 64) - 1)
            a_leader = 2 * a_linear_component * r1_leader - a_linear_component
            c_leader = 2 * r2_leader
            p = int.from_bytes(os.urandom(8), byteorder="big") / ((1 << 64) - 1)
            for j in range(len(self.min_values)):
                if p < 0.5:
                    if abs(a_leader) >= 1:
                        rand = int.from_bytes(os.urandom(8), byteorder="big") / ((1 << 64) - 1)
                        rand_leader_index = math.floor(position.shape[0] * rand)
                        x_rand = position[rand_leader_index, :]
                        distance_x_rand = abs(c_leader * x_rand[j] - position[i, j])
                        position[i, j] = np.clip(x_rand[j] - a_leader * distance_x_rand, self.min_values[j], self.max_values[j])
                    elif abs(a_leader) < 1:
                        distance_leader = abs(c_leader * leader[0, j] - position[i, j])
                        position[i, j] = np.clip(leader[0, j] - a_leader * distance_leader, self.min_values[j], self.max_values[j])
                elif p >= 0.5:
                    distance_leader = abs(leader[0, j] - position[i, j])
                    rand = int.from_bytes(os.urandom(8), byteorder="big") / ((1 << 64) - 1)
                    m_param = (b_linear_component - 1) * rand + 1
                    position[i, j] = np.clip(distance_leader * math.exp(self.spiral_param * m_param) * math.cos(m_param * 2 * math.pi) + leader[0, j], self.min_values[j], self.max_values[j])
            position[i, -1] = self.evaluate(position[i, 0:position.shape[1] - 1])
        return position

    def run(self):
        position = self.initial_position()
        leader = self.leader_position()
        iteration = 0
        max_iterations = 20000  # Set a maximum iteration limit to prevent infinite loop
        while np.abs(leader[0, -1] + self.qubits - 1) >= self.tol and iteration < max_iterations:
            if iteration % 100 == 1:
                print(f"Iteration: {iteration}, Best Fitness: {leader[0, -1]}, Evaluations: {self.func_eval_count}")
            a_linear_component = 2 - iteration * (2 / 50)
            a_linear_component = 2 - iteration * (2 / max_iterations)
            b_linear_component = -1 + iteration * (-1 / max_iterations)
            leader = self.update_leader(position, leader)
            position = self.update_position(position, leader, a_linear_component=a_linear_component, b_linear_component=b_linear_component)
            iteration += 1
            # Randomly reinitialize a portion of the population
            if iteration % 50 == 0:
                num_reinit = int(0.1 * self.hunting_party)
                for i in range(num_reinit):
                    for j in range(len(self.min_values)):
                        position[i, j] = random.uniform(self.min_values[j], self.max_values[j])
                    position[i, -1] = self.evaluate(position[i, 0:position.shape[1] - 1])
        print(f"Iteration: {iteration}, Best Fitness: {leader[0, -1]}, Evaluations: {self.func_eval_count}")
        return leader, self.func_eval_count



In [ ]:
min_qubits = 3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    woa = WhaleOptimizationAlgorithm(hunting_party=15, spiral_param=2, min_values=[-50]*dimension, max_values=[50]*dimension, 
                                     tol=1e-1, target_function=evaluate_expectation, dimension=dimension, qubits=qubits)
    best_leader, eval_count = woa.run()
    #print('Best leader:', best_leader)
    print('Function evaluations:', eval_count)


ansatz_num_parameters= 12


C:\Users\petre\AppData\Local\Temp\ipykernel_24080\1764180389.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  position[i, -1] = self.evaluate(position[i, 0:position.shape[1]-1])
C:\Users\petre\AppData\Local\Temp\ipykernel_24080\1764180389.py:35: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  leader[0, -1] = self.evaluate(leader[0, 0:leader.shape[1]-1])
C:\Users\petre\AppData\Local\Temp\ipykernel_24080\1764180389.py:68: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

Iteration: 1, Best Fitness: -0.65625, Evaluations: 31


C:\Users\petre\AppData\Local\Temp\ipykernel_24080\1764180389.py:91: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  position[i, -1] = self.evaluate(position[i, 0:position.shape[1] - 1])


Iteration: 101, Best Fitness: -1.5625, Evaluations: 1533
Iteration: 121, Best Fitness: -1.90625, Evaluations: 1833
Function evaluations: 1833
ansatz_num_parameters= 16
Iteration: 1, Best Fitness: -1.0, Evaluations: 31
Iteration: 101, Best Fitness: -2.0, Evaluations: 1533
Iteration: 201, Best Fitness: -2.65625, Evaluations: 3035
Iteration: 301, Best Fitness: -2.71875, Evaluations: 4537
Iteration: 401, Best Fitness: -2.8125, Evaluations: 6039
Iteration: 501, Best Fitness: -2.875, Evaluations: 7541
Iteration: 601, Best Fitness: -2.875, Evaluations: 9043
Iteration: 652, Best Fitness: -2.90625, Evaluations: 9809
Function evaluations: 9809
ansatz_num_parameters= 20
Iteration: 1, Best Fitness: -0.90625, Evaluations: 31
Iteration: 101, Best Fitness: -1.84375, Evaluations: 1533
Iteration: 201, Best Fitness: -2.28125, Evaluations: 3035
Iteration: 301, Best Fitness: -2.4375, Evaluations: 4537
Iteration: 401, Best Fitness: -3.375, Evaluations: 6039
Iteration: 501, Best Fitness: -3.375, Evaluations

KeyboardInterrupt: 